# 🎓 Day 3 — NLP (DistilBERT) | Arjun Pandey
## AI-Powered Student Performance & Dropout Predictor
### FDP 5-Day Academic AI Project

**Goal:** Fine-tune DistilBERT on student feedback text to classify concern type + urgency level. Add spaCy keyword extraction.

**Output:** `analyse_feedback('I am failing all subjects...')` → `{'concern': 'ACADEMIC', 'urgency': 'IMMEDIATE', 'keywords': ['backlog', 'attendance']}`

---
**Primary dataset:** 20 labelled student feedback samples (4 concern categories) — created here  
**Secondary dataset:** `dair-ai/emotion` from HuggingFace — for urgency detection  
**Model:** `distilbert-base-uncased` fine-tuned in <10 min on T4 GPU  
**Classes:** ACADEMIC (0) | FINANCIAL (1) | PERSONAL (2) | MENTAL_HEALTH (3)


In [1]:
# Cell 1 — Verify GPU
import torch
import tensorflow as tf
print('CUDA available:', torch.cuda.is_available())
print('TF GPU devices:', tf.config.list_physical_devices('GPU'))
!nvidia-smi

CUDA available: True
TF GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Fri Apr 10 07:09:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                   

In [2]:
# Cell 2 — Install packages
!pip install -q datasets transformers huggingface_hub accelerate
!pip install -q transformers datasets torch accelerate spacy
!python -m spacy download en_core_web_sm
print('All packages installed ✓')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 119.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All packages installed ✓


In [3]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/fdp_student_ai', exist_ok=True)
SAVE_DIR = '/content/drive/MyDrive/fdp_student_ai'
print('Drive mounted. Save directory ready:', SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Save directory ready: /content/drive/MyDrive/fdp_student_ai


## 📝 Cell 4 — Create Labelled Student Feedback Dataset

20 hand-labelled samples across 4 concern categories. We'll expand to 80+ by paraphrasing.

In [4]:
from datasets import Dataset, DatasetDict
import pandas as pd

# Student feedback samples — concern type labels
# 0=ACADEMIC  1=FINANCIAL  2=PERSONAL  3=MENTAL_HEALTH
student_data = [
    # ACADEMIC (0)
    ('I am not able to understand the subjects and failing every test', 0),
    ('My attendance is very low because I cannot focus in class', 0),
    ('I have failed 3 backlogs and do not know what to do', 0),
    ('Cannot keep up with assignments and lab work simultaneously', 0),
    ('Professors teach too fast and I fall behind every week', 0),
    ('My grades are dropping and I am scared of the semester results', 0),
    ('I missed too many classes and now I cannot appear in the exam', 0),
    # FINANCIAL (1)
    ('My fee payment is pending and I may not be able to appear in exam', 1),
    ('I cannot afford the hostel fee this semester', 1),
    ('No money for buying textbooks and lab materials', 1),
    ('Family financial crisis is affecting my studies badly', 1),
    ('I need scholarship urgently or I will have to drop this year', 1),
    # PERSONAL (2)
    ('Fighting with roommates and cannot concentrate on studies', 2),
    ('Family problems are distracting me from college work', 2),
    ('Relationship issues are affecting my performance in class', 2),
    ('I feel very lonely and have no friends in college', 2),
    # MENTAL_HEALTH (3)
    ('I am feeling very anxious and stressed about everything', 3),
    ('Cannot sleep at night and keep thinking about failing college', 3),
    ('I feel like giving up everything and leaving college', 3),
    ('Depression is making it impossible to attend classes regularly', 3),
    ('I have panic attacks before every examination this semester', 3),
    ('Feeling completely hopeless about my academic future', 3),
]

LABEL_MAP = {0: 'ACADEMIC', 1: 'FINANCIAL', 2: 'PERSONAL', 3: 'MENTAL_HEALTH'}

texts  = [d[0] for d in student_data]
labels = [d[1] for d in student_data]

df_feedback = pd.DataFrame({'text': texts, 'label': labels})
df_feedback['concern'] = df_feedback['label'].map(LABEL_MAP)

print(f'Base dataset: {len(df_feedback)} labelled samples')
print(df_feedback.groupby('concern').size())
print()
print(df_feedback.head(8).to_string())

Base dataset: 22 labelled samples
concern
ACADEMIC         7
FINANCIAL        5
MENTAL_HEALTH    6
PERSONAL         4
dtype: int64

                                                                text  label    concern
0    I am not able to understand the subjects and failing every test      0   ACADEMIC
1          My attendance is very low because I cannot focus in class      0   ACADEMIC
2                I have failed 3 backlogs and do not know what to do      0   ACADEMIC
3        Cannot keep up with assignments and lab work simultaneously      0   ACADEMIC
4             Professors teach too fast and I fall behind every week      0   ACADEMIC
5     My grades are dropping and I am scared of the semester results      0   ACADEMIC
6      I missed too many classes and now I cannot appear in the exam      0   ACADEMIC
7  My fee payment is pending and I may not be able to appear in exam      1  FINANCIAL


## 🔁 Cell 4b — Expand to 80+ Samples via Paraphrasing

In [5]:
# Expand dataset by creating paraphrased variations
paraphrase_map = {
    0: [
        'My exam scores are very poor and I keep failing internal tests',
        'I do not understand the syllabus and my attendance is dropping',
        'I have multiple failed subjects and am worried about my future',
        'I struggle to complete lab assignments and written tests together',
        'The pace of teaching is too fast for me and I am falling behind',
        'I am unable to score passing marks in any of my subjects',
        'My cumulative performance this semester has been terrible',
    ],
    1: [
        'I have not been able to pay my examination fee and need help',
        'The hostel charges are too high and I cannot manage this month',
        'I cannot buy required books because of financial constraints',
        'Financial problems at home are badly affecting my study schedule',
        'Without a scholarship I will be forced to leave college this year',
        'I borrowed money for fees but still could not pay the full amount',
    ],
    2: [
        'Issues with hostel roommates are causing me great distraction',
        'Problems in my family are stopping me from concentrating on studies',
        'A personal relationship problem is hurting my academic focus',
        'I feel isolated in college and it is affecting my motivation',
        'Conflicts with peers are making college life very stressful',
    ],
    3: [
        'I constantly feel nervous and overwhelmed with college pressure',
        'I cannot sleep well and overthink about my academic failures',
        'I feel like quitting college because nothing is going right',
        'Depressive episodes are preventing me from coming to class',
        'Severe exam anxiety is affecting my ability to prepare properly',
        'I have completely lost hope that I can pass this semester',
        'The pressure of studies has made me mentally exhausted',
    ],
}

aug_texts, aug_labels = list(texts), list(labels)
for label_id, paraphrases in paraphrase_map.items():
    aug_texts.extend(paraphrases)
    aug_labels.extend([label_id] * len(paraphrases))

print(f'Expanded dataset: {len(aug_texts)} samples')
for k, v in LABEL_MAP.items():
    count = aug_labels.count(k)
    print(f'  {v}: {count} samples')

Expanded dataset: 47 samples
  ACADEMIC: 14 samples
  FINANCIAL: 11 samples
  PERSONAL: 9 samples
  MENTAL_HEALTH: 13 samples


## 🤗 Cell 5 — Load `dair-ai/emotion` for Urgency Detection

Maps fear/anger/sadness → IMMEDIATE | joy/love/surprise → LOW

## 🧠 Cell 6 — Fine-Tune DistilBERT for Concern Classification

In [13]:
!pip install --upgrade transformers accelerate

In [14]:
import transformers
print(transformers.__version__)

5.5.3


In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch
import numpy as np

MODEL_ID = 'distilbert-base-uncased'
print(f'Loading tokenizer: {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

class FeedbackDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc    = tokenizer(texts, padding=True, truncation=True, max_length=128)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item

# Train/test split
tr_t, va_t, tr_l, va_l = train_test_split(
    aug_texts, aug_labels, test_size=0.2, random_state=42, stratify=aug_labels
)

train_ds = FeedbackDataset(tr_t, tr_l)
val_ds   = FeedbackDataset(va_t, va_l)

print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)}')

# Load DistilBERT for 4-class classification
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=4)

# Training arguments
# args = TrainingArguments(
#     # output_dir='./nlp_model',
#     # num_train_epochs=12,
#     # per_device_train_batch_size=8,
#     # # evaluation_strategy='epoch',
#     # # save_strategy='epoch',
#     # # load_best_model_at_end=True,
#     # # metric_for_best_model='accuracy',
#     # do_eval=True,
#     # logging_steps=5,
#     # # logging_steps=5,
#     # no_cuda=False,
#     # report_to='none',

#     output_dir='./nlp_model',
#     num_train_epochs=4,  # small dataset
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     evaluation_strategy='epoch',
#     save_strategy='epoch',
#     load_best_model_at_end=True,
#     metric_for_best_model='accuracy',
#     logging_steps=5,
#     report_to='none'
# )
args = TrainingArguments(
    output_dir='./nlp_model',
    num_train_epochs=3,
    per_device_train_batch_size=8
)
def compute_metrics(p):
    return {'accuracy': accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))}

trainer = Trainer(
    model=model_bert,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

print('\nFine-tuning DistilBERT on T4 GPU (6–10 minutes)...')
trainer.train()
print('\n✅ Fine-tuning complete!')

Loading tokenizer: distilbert-base-uncased...
Train samples: 37 | Val samples: 10


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Fine-tuning DistilBERT on T4 GPU (6–10 minutes)...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Fine-tuning complete!


## 🔍 Cell 7 — spaCy Keyword Extractor

In [17]:
import spacy

nlp_spacy = spacy.load('en_core_web_sm')

# Domain-specific concern keywords per category
CONCERN_WORDS = {
    'ACADEMIC':      ['attendance', 'exam', 'fail', 'backlog', 'assignment', 'grade', 'subject', 'marks', 'test'],
    'FINANCIAL':     ['fee', 'money', 'scholarship', 'afford', 'payment', 'hostel', 'expense', 'borrow'],
    'PERSONAL':      ['family', 'roommate', 'relationship', 'friend', 'lonely', 'distract', 'conflict'],
    'MENTAL_HEALTH': ['anxious', 'stress', 'depression', 'hopeless', 'sleep', 'panic', 'lonely', 'overwhelm'],
}

def extract_keywords(text, concern_label):
    """
    Extract concern-relevant keywords from student feedback text.
    First tries domain-specific words, then falls back to spaCy nouns/verbs.
    """
    doc  = nlp_spacy(text.lower())
    base = CONCERN_WORDS.get(concern_label, [])
    found = [w for w in base if w in text.lower()]

    if found:
        return found[:5]
    else:
        # Fallback: nouns and verbs from spaCy (excluding stopwords)
        return [t.text for t in doc if not t.is_stop and t.pos_ in ('NOUN', 'VERB')][:4]

# Test keyword extractor
print('Keyword extractor tests:')
print(extract_keywords('I am failing exams and my attendance is very low', 'ACADEMIC'))
print(extract_keywords('Cannot pay hostel fee and may have to leave college', 'FINANCIAL'))
print(extract_keywords('I feel hopeless and cannot sleep due to anxiety', 'MENTAL_HEALTH'))

Keyword extractor tests:
['attendance', 'exam', 'fail']
['fee', 'hostel']
['hopeless', 'sleep']


## 💾 Cell 8 — Save Model & Write `analyse_feedback()` Function

In [20]:
import os

# Save fine-tuned DistilBERT + tokenizer
nlp_model_path = f'{SAVE_DIR}/nlp_model'
trainer.save_model(nlp_model_path)
tokenizer.save_pretrained(nlp_model_path)

print(f'✅ NLP model saved to: {nlp_model_path}')

# Urgency detection keywords
URGENCY_KEYWORDS = [
    'fail', 'urgent', 'drop', 'leave', 'hopeless', 'crisis',
    'cannot', 'panic', 'quit', 'impossible', 'helpless', 'give up'
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert.to(device)
model_bert.eval()   # optional but recommended
def analyse_feedback(text):
    """
    Analyse student feedback text for concern type, urgency, and keywords.

    Args:
        text : Free-text student feedback string

    Returns:
        dict: {'concern': 'ACADEMIC', 'urgency': 'IMMEDIATE', 'keywords': [...]}
    '''
    # Encode and predict concern type
    '''enc = tokenizer([text], padding=True, truncation=True, max_length=128, return_tensors='pt')

    with torch.no_grad():
        logits = model_bert(**enc).logits

    pred_id = int(torch.argmax(logits))
    concern = LABEL_MAP[pred_id]

    # Urgency scoring by keyword matching
    urg_score = sum(1 for w in URGENCY_KEYWORDS if w in text.lower())
    urgency = 'IMMEDIATE' if urg_score >= 2 else 'MODERATE' if urg_score == 1 else 'LOW'

    # Keyword extraction
    keywords = extract_keywords(text, concern)

    return {'concern': concern, 'urgency': urgency, 'keywords': keywords}
"""

    enc = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    # 👉 Move inputs to GPU
    enc = {k: v.to(device) for k, v in enc.items()}

    # 👉 Inference
    with torch.no_grad():
        outputs = model_bert(**enc)
        logits = outputs.logits

    pred_id = int(torch.argmax(logits, dim=1))
    concern = LABEL_MAP[pred_id]

    # Urgency scoring
    urg_score = sum(1 for w in URGENCY_KEYWORDS if w in text.lower())
    urgency = 'IMMEDIATE' if urg_score >= 2 else 'MODERATE' if urg_score == 1 else 'LOW'

    keywords = extract_keywords(text, concern)

    return {
        'concern': concern,
        'urgency': urgency,
        'keywords': keywords
    }

# ── Live tests ────────────────────────────────────────────────────────
print('=== Live analyse_feedback() Tests ===')
print()
print('Test 1: Academic concern')
print(analyse_feedback('I am failing all subjects and my attendance is very low'))

print()
print('Test 2: Financial concern')
print(analyse_feedback('Cannot pay hostel fee and may have to leave college'))

print()
print('Test 3: Mental health — IMMEDIATE')
print(analyse_feedback('I feel hopeless and cannot sleep thinking about failing and want to give up'))

print()
print('Test 4: Personal concern')
print(analyse_feedback('Family problems and roommate conflicts are distracting me from studies'))

print()
print('Test 5: Mixed concern')
print(analyse_feedback('I am failing exams and cannot attend college due to stress and personal issues'))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ NLP model saved to: /content/drive/MyDrive/fdp_student_ai/nlp_model
=== Live analyse_feedback() Tests ===

Test 1: Academic concern
{'concern': 'ACADEMIC', 'urgency': 'MODERATE', 'keywords': ['attendance', 'fail', 'subject']}

Test 2: Financial concern
{'concern': 'ACADEMIC', 'urgency': 'IMMEDIATE', 'keywords': ['pay', 'hostel', 'fee', 'leave']}

Test 3: Mental health — IMMEDIATE
{'concern': 'MENTAL_HEALTH', 'urgency': 'IMMEDIATE', 'keywords': ['hopeless', 'sleep']}

Test 4: Personal concern
{'concern': 'ACADEMIC', 'urgency': 'LOW', 'keywords': ['family', 'problems', 'roommate', 'conflicts']}

Test 5: Mixed concern
{'concern': 'ACADEMIC', 'urgency': 'IMMEDIATE', 'keywords': ['exam', 'fail']}


In [21]:
# Cell 9 — Export nlp_module.py for Piyush (Day 5)

nlp_module_code = '''
# nlp_module.py — Day 3 deliverable (Arjun Pandey)
# Usage: from nlp_module import analyse_feedback

import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = '/content/drive/MyDrive/fdp_student_ai'
NLP_MODEL_PATH = f'{SAVE_DIR}/nlp_model'

LABEL_MAP = {0: 'ACADEMIC', 1: 'FINANCIAL', 2: 'PERSONAL', 3: 'MENTAL_HEALTH'}
URGENCY_KEYWORDS = ['fail', 'urgent', 'drop', 'leave', 'hopeless', 'crisis', 'cannot', 'panic', 'quit', 'impossible']
CONCERN_WORDS = {
    'ACADEMIC': ['attendance', 'exam', 'fail', 'backlog', 'assignment', 'grade', 'subject'],
    'FINANCIAL': ['fee', 'money', 'scholarship', 'afford', 'payment', 'hostel'],
    'PERSONAL': ['family', 'roommate', 'relationship', 'friend', 'lonely'],
    'MENTAL_HEALTH': ['anxious', 'stress', 'depression', 'hopeless', 'sleep', 'panic'],
}

_tokenizer = AutoTokenizer.from_pretrained(NLP_MODEL_PATH)
_model_bert = AutoModelForSequenceClassification.from_pretrained(NLP_MODEL_PATH)
_nlp_spacy = spacy.load('en_core_web_sm')

def extract_keywords(text, concern_label):
    doc = _nlp_spacy(text.lower())
    base = CONCERN_WORDS.get(concern_label, [])
    found = [w for w in base if w in text.lower()]
    return found[:5] if found else [t.text for t in doc if not t.is_stop and t.pos_ in ('NOUN','VERB')][:4]

def analyse_feedback(text):
    enc = _tokenizer([text], padding=True, truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        logits = _model_bert(**enc).logits
    pred_id = int(torch.argmax(logits))
    concern = LABEL_MAP[pred_id]
    urg_score = sum(1 for w in URGENCY_KEYWORDS if w in text.lower())
    urgency = 'IMMEDIATE' if urg_score >= 2 else 'MODERATE' if urg_score == 1 else 'LOW'
    keywords = extract_keywords(text, concern)
    return {'concern': concern, 'urgency': urgency, 'keywords': keywords}
'''

with open(f'{SAVE_DIR}/nlp_module.py', 'w') as f:
    f.write(nlp_module_code)

print(f'✅ nlp_module.py saved to {SAVE_DIR}')
print()
print('Day 3 Complete! Files delivered to Drive:')
print('  ✅ nlp_model/ (DistilBERT fine-tuned folder)')
print('  ✅ nlp_module.py')

✅ nlp_module.py saved to /content/drive/MyDrive/fdp_student_ai

Day 3 Complete! Files delivered to Drive:
  ✅ nlp_model/ (DistilBERT fine-tuned folder)
  ✅ nlp_module.py
